# Step 27 — cross-platform: does the E1–E2 axis exist in the other studies?

**Data types: RNA_array, bulk_RNA_seq, scRNA_seq pseudobulk.** **Reads:** `step26_shared.rds`,
`step10_site_*`, `step23_GSE232381.rds`, `step25_GSE135779.rds`. **Writes:** `step27_preservation.rds`.

Before assigning endotypes in another study, we check whether the axis that separates them exists
there. The **axis score** of a sample is the mean standardised expression of the 30 E2 marker genes
minus that of the 30 E1 marker genes. If the axis is real in a study, E2 markers rise together with
the score and E1 markers fall with it.

Each study computes this locally and sends back only one number per marker gene: its correlation with
the local axis score. For GSE65391 only **validation** patients are used, because the markers were
chosen on discovery patients.

**The granulocyte test.** E2 is marked by neutrophil granule genes. PBMC preparations remove most
mature neutrophils, so these genes may be close to absent in GSE135779. Each study also reports
where each marker gene sits in its own expression distribution (percentile of mean expression). That
is a scale-free summary, comparable across platforms.

In [1]:
source("../src/paths.R")
source("../src/federation.R")
start_log("27")
sh <- readRDS(art("step26_shared.rds"))
mk <- intersect(sh$markers, sh$shared)
e1 <- intersect(mk, sh$markers[1:30]); e2 <- intersect(mk, sh$markers[31:60])
c(E1_markers = length(e1), E2_markers = length(e2))

E1_markers E2_markers 
        25         14

In [2]:
study_summary <- function(E, name, n) {
  Z <- t(scale(t(E[mk, ])))                                 # standardised within the study
  score <- colMeans(Z[e2, ]) - colMeans(Z[e1, ])
  pct <- rank(rowMeans(E)) / nrow(E)                         # percentile among all measured genes
  send(list(r = apply(Z, 1, cor, y = score), percentile = pct[mk]), name, "marker correlations and percentiles", n)
}
first_val <- function(m) { d <- m[m$split == "validation", ]; d <- d[order(d$subject, d$visit), ]; rownames(d)[!duplicated(d$subject)] }
res <- list()
for (s in SITES) { d <- readRDS(site_file("10", s)); j <- first_val(d$meta)
  res[[paste0("GSE65391_", s)]] <- study_summary(d$E[, j], s, length(j)) }
b <- readRDS(art("step23_GSE232381.rds")); res$GSE232381 <- study_summary(b$E, "GSE232381", ncol(b$E))
p <- readRDS(art("step25_GSE135779.rds")); j <- p$meta$disease == "SLE"
res$GSE135779 <- study_summary(p$E[, j], "GSE135779", sum(j))

In [3]:
tab <- t(sapply(res, function(x) c(
  E2_markers_median_r = median(x$r[e2]), E1_markers_median_r = median(x$r[e1]),
  E2_markers_median_percentile = median(x$percentile[e2]), E1_markers_median_percentile = median(x$percentile[e1]))))
round(tab, 2)

,E2_markers_median_r,E1_markers_median_r,E2_markers_median_percentile,E1_markers_median_percentile
GSE65391_A,0.56,-0.75,0.83,0.83
GSE65391_B,0.60,-0.79,0.87,0.83
GSE65391_C,0.65,-0.71,0.79,0.85
GSE232381,0.77,-0.75,0.49,0.70
GSE135779,0.51,-0.59,0.13,0.76


## The granulocyte genes, one by one

In [4]:
gran <- intersect(c("LTF", "ELANE", "MPO", "AZU1", "CTSG", "DEFA3", "DEFA4", "CAMP", "BPI", "CEACAM8", "LCN2", "OLFM4", "MMP9", "ARG1"), mk)
round(sapply(res, function(x) x$percentile[gran]), 2)
round(sapply(res, function(x) x$r[gran]), 2)

,GSE65391_A,GSE65391_B,GSE65391_C,GSE232381,GSE135779
ELANE,0.82,0.86,0.77,0.31,0.13
MPO,0.67,0.71,0.65,0.45,0.16
AZU1,0.66,0.70,0.63,0.29,0.22
CAMP,0.95,0.97,0.95,0.57,0.07
BPI,0.73,0.79,0.72,0.50,0.05
LCN2,0.94,0.96,0.94,0.73,0.13
MMP9,0.96,0.96,0.95,0.89,0.08


,GSE65391_A,GSE65391_B,GSE65391_C,GSE232381,GSE135779
ELANE,0.41,0.63,0.91,0.77,0.72
MPO,0.51,0.58,0.78,0.71,0.61
AZU1,0.57,0.55,0.60,0.79,0.73
CAMP,0.82,0.72,0.61,0.78,0.54
BPI,0.55,0.74,0.79,0.72,0.49
LCN2,0.66,0.75,0.67,0.77,0.38
MMP9,0.86,0.50,0.61,0.93,0.17


In [5]:
saveRDS(list(summary = tab, per_gene = res), art("step27_preservation.rds"))

## Findings

- **The E1–E2 axis exists in all three studies.** E2 markers rise with the axis score (median r +0.51
  to +0.77) and E1 markers fall with it (−0.59 to −0.79), on both platforms and in both tissues. In
  GSE65391 this holds on validation patients, who played no part in choosing the markers.
- **In PBMC the granulocyte genes are nearly silent, but not flat.** In GSE135779 they sit at the 5th
  to 22nd percentile of expression, against the 63rd to 97th in whole blood. Their levels still vary
  together with the axis (r 0.17 to 0.73). This is what low-density granulocytes in lupus PBMC would
  produce. The axis is present in PBMC, but it is carried more by the lymphocyte side than by the
  granulocyte side.
- GSE232381 sits in between (29th to 89th percentile). Its "peripheral blood cells" are closer to
  whole blood than to PBMC.